In [1]:

from micrograd.value import ScalarValue
import random

class Neuron:
    
    def __init__(self, nin):
        self.w = [ScalarValue(random.uniform(-1,1)) for _ in range(nin)]
        self.b = ScalarValue(random.uniform(-1,1))
        

    def __call__(self, x):
        # w * x + b
        act = sum((wi*xi for wi, xi in zip(self.w, x)), self.b)
        out = act.tanh()
        return out
    
    def parameters(self):
        return self.w + [self.b]

    def __repr__(self):
        return f"Neuron(w={self.w}, b={self.b})"
    

In [33]:
n1 = Neuron(2)
x = [2.0, 3.0]

n1(x)

ScalarValue(data=-0.02076385675577734, grad=0)

In [2]:

class Layer:
    
    def __init__(self, nin, nout):
        self.neurons = [Neuron(nin) for _ in range(nout)]
    
    def __call__(self, x):
        outs = [neuron(x) for neuron in self.neurons]
        return outs[0] if len(outs) ==1 else outs
    
    def parameters(self):
        return [p for neuron in self.neurons for p in neuron.parameters()]
        

In [55]:
layer1 = Layer(2, 3)
x = [2,4]

layer1_out = layer1(x)
layer1_out

hidden_layer1 = Layer(3, 4)
hidden_layer1_out = hidden_layer1(layer1_out)
hidden_layer1_out


[ScalarValue(data=0.5345193717060603, grad=0),
 ScalarValue(data=-0.4274682075398252, grad=0),
 ScalarValue(data=0.10303546728661597, grad=0),
 ScalarValue(data=-0.9514973003888298, grad=0)]

In [3]:

class MLP:
    
    def __init__(self, nin, nouts):
        sz = [nin] + nouts
        self.layers = [Layer(nin=sz[i], nout=sz[i+1]) for i in range(len(nouts))]
    
    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x
    
    def parameters(self):
        return [ p for layer in self.layers for p in layer.parameters()]

x = [3.0,4.0,-24.0]
mlp1 = MLP(3, [4,4,1])

mlp1(x)


ScalarValue(data=0.5721903463302923, grad=0)

In [ ]:
mlp1.parameters()

In [4]:
xs = [
    [2.0, 3.0, -1.0],
    [3.0, -1.0, 0.5],
    [0.5, 1.0, 1.0],
    [1.0, 1.0, -1.0],
] # dataset

ys = [1.0, -1.0, -1.0, 1.0]  # desired targets

# try in above MLP
ypred = [mlp1(x) for x in xs]
ypred

[ScalarValue(data=0.4381556433965434, grad=0),
 ScalarValue(data=0.3859797332154174, grad=0),
 ScalarValue(data=0.5606230674259671, grad=0),
 ScalarValue(data=0.5322462260932364, grad=0)]

In [5]:
loss = sum([(ypredi - ysi)**2 for ysi, ypredi in zip(ys, ypred)])
loss

ScalarValue(data=4.890946853517087, grad=0)

In [6]:
print(f"first layer first neuron weight :: before backprop : {mlp1.layers[0].neurons[0].w[0]}")

first layer first neuron weight :: before backprop : ScalarValue(data=-0.6411099456262495, grad=0)


In [13]:
loss.backward()

In [8]:
print(f"first layer first neuron weight :: after backprop : {mlp1.layers[0].neurons[0].w[0]}")

first layer first neuron weight :: after backprop : ScalarValue(data=-0.6411099456262495, grad=0.023165503734989337)


In [14]:
# update or optimize the parameters now
step_size = 0.01
for p in mlp1.parameters():
    p.data += -step_size * p.grad 
    """
    (- sign) : since the loss.backward indicate direction towards more loss 
    so: push data little step size factorized opposite of respective gradient.
    """


In [15]:
print(f"first layer first neuron weight :: after parameters updates : {mlp1.layers[0].neurons[0].w[0]}")

first layer first neuron weight :: after parameters updates : ScalarValue(data=-0.6417711353606748, grad=0.042953469707537176)


In [16]:
# now re-calculate the forward and loss : see the difference in loss value
ypred = [mlp1(x) for x in xs]
loss = sum([(ypredi - ysi)**2 for ysi, ypredi in zip(ys, ypred)])
print(f"ypred: {ypred} \nloss:{loss}")

ypred: [ScalarValue(data=0.26737773407017146, grad=0), ScalarValue(data=0.07620914793774194, grad=0), ScalarValue(data=0.32425389639469343, grad=0), ScalarValue(data=0.39336036083806497, grad=0)] 
loss:ScalarValue(data=3.816621548560087, grad=0)


In [ ]:
from micrograd.optimizer import optimize_params
# iterate these steps for N times see the loss value
N = 100 
for i in range(N):
    
    # 1. forward pass
    ypred = [mlp1(x) for x in xs]
    
    # 2. loss
    loss = sum([(ypredi-ysi)**2 for ysi, ypredi in zip(ys, ypred)])
    if i % 10 == 0:
        print(f"iteration: {i} :: loss = {loss}")
    
    # 3. backpropagation
    # zero the grad before backprop
    for p in mlp1.parameters():
        p.grad = 0.0
    loss.backward()
    
    # 4. update or optimize the parameters 
    # for p in mlp1.parameters():
    #     p.data += -0.01 * p.grad
    optimize_params(learning_rate=0.01, params=mlp1.parameters())
    

print(f"ypred[0]:{ypred[0]}\nys[0]:{ys[0]}\nloss:{loss}")
    

iteration: 0 :: loss = ScalarValue(data=0.0, grad=0)
iteration: 10 :: loss = ScalarValue(data=0.0, grad=0)
iteration: 20 :: loss = ScalarValue(data=0.0, grad=0)
iteration: 30 :: loss = ScalarValue(data=0.0, grad=0)
iteration: 40 :: loss = ScalarValue(data=0.0, grad=0)
iteration: 50 :: loss = ScalarValue(data=0.0, grad=0)
iteration: 60 :: loss = ScalarValue(data=0.0, grad=0)
iteration: 70 :: loss = ScalarValue(data=0.0, grad=0)
iteration: 80 :: loss = ScalarValue(data=0.0, grad=0)
iteration: 90 :: loss = ScalarValue(data=0.0, grad=0)
ypred[0]:ScalarValue(data=1.0, grad=0.0)
ys[0]:1.0
loss:ScalarValue(data=0.0, grad=1)


In [22]:

print(f"ypred:{ypred}\nys:{ys}\nloss:{loss}")

ypred:[ScalarValue(data=1.0, grad=0.0), ScalarValue(data=-1.0, grad=0.0), ScalarValue(data=-1.0, grad=0.0), ScalarValue(data=1.0, grad=0.0)]
ys:[1.0, -1.0, -1.0, 1.0]
loss:ScalarValue(data=0.0, grad=1)
